In [1]:
import sys
import os
import pandas as pd

src_path = os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

from dense_index_bge import DenseIndex
from sparse_index import SparseIndex
import citation_utils
import metric_utils
import reranker_utils
import rrf
import hits_utils


libgomp: Invalid value for environment variable OMP_NUM_THREADS


In [2]:
court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = dict(zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()))

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

court_doc = [{'citation':citation, 'text':text} for citation,text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist())]
law_doc = [{'citation':citation, 'text':text} for citation,text in zip(law_df['citation'].tolist(), law_df['text'].tolist())]

test_df = pd.read_csv("../data/test_rewrite_001.csv")
_d = {}
for _, row in test_df.iterrows():
    if row['query_id'] not in _d:
        _d[row['query_id']] = [row['query']]
    else:
        _d[row['query_id']].append(row['query'])
test_dict = {k: v for k, v in sorted(_d.items())}

valid_df = pd.read_csv("../data/valid_rewrite_001.csv")

print("data loaded.")

data loaded.


In [3]:
# BGE-embedding

from FlagEmbedding import FlagReranker, BGEM3FlagModel

model = BGEM3FlagModel('/root/.cache/modelscope/hub/models/BAAI/bge-m3', use_fp16=True, show_progress_bar=False)

dense_index = DenseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)

sparse_index = SparseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)

# reranker = FlagReranker('../ft_data/merged_reranker', use_fp16=True, normalize=True) # Setting use_fp16 to True speeds up computation with a slight performance degradation
reranker = FlagReranker('/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3', use_fp16=True, normalize=True)


libgomp: Invalid value for environment variable OMP_NUM_THREADS

libgomp: Invalid value for environment variable OMP_NUM_THREADS
/root/miniconda3/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/root/miniconda3/compiler_compat/ld: warning: librt.so.1, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libpthread.so.0, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `std::runtime_error::~runtime_error()@GLIBCXX_3.4

DenseIndex.embeddings:  (2107648, 1024)


In [7]:
%load_ext autoreload

%autoreload 2

from pipeline import Pipeline

p = Pipeline(court_consideration_df,
             court_consideration_d,
             law_df,
             law_d,
             dense_index,
             sparse_index,
             reranker,
             test_df,
             valid_df,
             citation_agg_w1=0.2,
             citation_agg_w2=0.7,
             citation_agg_w3=0.5,
             global_citaion_ranking_pool_method='sum'
            )

p.evaluate()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


evaluation:   0%|          | 0/10 [00:00<?, ?it/s]

[recall] hit_with_score_l.len: 1612
[normalize_sr] count: 34
[rerank] sliced: 1402/12225
[5] r: 0.023809523809523808 p: 0.2 f1: 0.0425531914893617
[10] r: 0.023809523809523808 p: 0.1 f1: 0.038461538461538464
[15] r: 0.07142857142857142 p: 0.2 f1: 0.10526315789473682
[20] r: 0.09523809523809523 p: 0.2 f1: 0.12903225806451613
[25] r: 0.09523809523809523 p: 0.16 f1: 0.11940298507462685
[30] r: 0.11904761904761904 p: 0.16666666666666666 f1: 0.1388888888888889
[35] r: 0.11904761904761904 p: 0.14285714285714285 f1: 0.12987012987012989
[40] r: 0.11904761904761904 p: 0.125 f1: 0.12195121951219512
[45] r: 0.11904761904761904 p: 0.1111111111111111 f1: 0.11494252873563218
